In [ ]:
!service mongodb start

In [ ]:
%env FW_CONFIG_FILE=/root/capsule/code/config/FW_config.yaml

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Molecular Dynamics Simulations

So far, you have learned the basics of running DFT workflows in MISPR. We will now move on to running MD simulations in an automated manner by utilizing the workflows in MISPR. 

The goal is to model the behavior of 0.1 M TBA⁺, BF₄⁻ salt and 0.1 M CO₂ in Propionitrile solvent. We will go through both how to set up the input files, as well as how to set up the simulations to minimize and equilibrate the system. Finally we briefly introduce some of the MISPR tools for analyzing molecular simulation trajectories.

# System composition

In [ ]:
from IPython.display import Image

Image("/root/capsule/data/md/composition.png", width=600, height=600)

## Structure inputs and initial configuration

The starting point for each simulation is the structure file for each component of the liquid solution. For this tutorial, we will use the structure of TBA⁺, BF₄⁻, CO₂, Propionitrile in the xyz file format. 
From the individual structure files and the number of components of each type, MISPR randomly packs these molecules in the box of the specified length (7 nm here) to prepare an initial configuration to be used in the simulations. 

Let us take a peek at the individual structure files and a visualization of the generated configuration. 

In [ ]:
from pymatgen.core.structure import Molecule 

bf4_mol = Molecule.from_file("/root/capsule/data/md/bf4.xyz")
bf4_mol.set_charge_and_spin(-1, 1)
print(f"bf4 structure\n-------------\n{bf4_mol}")
print("-----------------------------------")
tba_mol = Molecule.from_file("/root/capsule/data/md/tba.xyz")
tba_mol.set_charge_and_spin(1, 1)
print(f"tba structure\n-------------\n{tba_mol}")
print("-----------------------------------")
co2_mol = Molecule.from_file("/root/capsule/data/md/co2.xyz")
print(f"co2 structure\n-------------\n{co2_mol}")
print("-----------------------------------")
propionitrile_mol = Molecule.from_file("/root/capsule/data/md/propionitrile.xyz")
print(f"propionitrile structure\n-------------\n{propionitrile_mol}")

In [ ]:
Image("/root/capsule/data/md/mixture.png", width=500, height=500)

# Data file

To model the atomistic interactions, we need to obtain the force field parameters.

These parameters include nonbonded parameters (atom types and charges) as well as bonded parameters (bonds, angles, dihedrals and atom connectivity).
MISPR supports automatic generation of these parameters through AmberTools (http://ambermd.org) or automatic retrieval from our database. Using these parameters, it generates a data file that contains all the information about the system to run the simulations. To summarize, this information represents:

1. Configuration of the system (i.e., initial atomic positions)
2. Box size
3. Molecular topology (i.e., which atoms are bonded)
4. Force field parameters

In this tutorial, we will directly provide the data file as an input to the workflow instead of asking it to generate it. Generation of the data file is typically done within the workflow and the user only needs to provide the structure of the individual molecules and the composition (i.e, concentration and system dimension) as inputs. However, here we will skip this step so the workflow will not require AmberTools (which if is installed within the capsule, will increase the time required for building the enviornment). 

**Note**: The data file we provide here has been generated before using the same workflow from a previous run.
Now, let's take a look at the first few lines of the data file.

In [ ]:
with open("/root/capsule/data/md/data.mixture") as data_file:
    print(data_file.read(1008))

## Running the MD workflow

Now, we are ready to import the MD workflow and create a workflow object for our electrolyte system. 

In [ ]:
from mispr.lammps.utilities.utilities import run_fake_lammps
from mispr.lammps.workflows.base import lammps_workflow

template_dir = "/root/capsule/data/md/templates"
ref_dir = "/root/capsule/data/md/runs"

molecules = [tba_mol, propionitrile_mol, bf4_mol, co2_mol]
mol_names = ["tba", "propionitrile", "bf4", "co2"]
num_mols = [20, 2625, 20, 20]
num_atoms_per_mol = [53, 9 ,5, 3]
mass = [
    14.006700,
    12.010700,
     1.007940,
    14.006700,
    12.010700,
    12.010700,
     1.007940,
    10.811000,
    18.998403,
    15.999400,
    12.010700 ]


wf = lammps_workflow(
    data_file_name="/root/capsule/data/md/data.mixture",
    working_dir="/root/capsule/results/md",
    recipe=[
        ["min", ["template_filename", "min.in"]],
        ["npt", ["template_filename", "npt.in"]],
        ["melt", ["template_filename", "melt.in"]],
        ["quench", ["template_filename", "quench.in"]],
        ["nvt", ["template_filename", "nvt.in"]],
    ],
    analysis_list=["diffusion", "rdf"],
    analysis_settings=[
        {
            "diff_names": mol_names,
            "timestep": 1,
            "com_drift": True,
            "msd_method": "from_dump",
            "dump_freq": 50000,
            "diff_dist": True,
            "avg_interval": True,
            "num_mols": num_mols,
            "num_atoms_per_mol": num_atoms_per_mol,
        },
        {
            "rdf_type": "molecular",
            "partial_relations": [[11,11,11], [1,2,3]],
            "use_default_atom_ids": True,
            "mass": mass,
            "num_mols": num_mols,                    
            "num_atoms_per_mol": num_atoms_per_mol
        },
    ],
    template_dir=template_dir,
)

The above workflow encapsulates all the steps required to fully automate not only the MD simulations but also the subsequent analysis. 

More specifically, here the workflow will run the following steps:
1. Minimize the system 
2. Equilibrate the system at constant number of particles (N), pressure (P), and temperature (T) - NPT run
3. Melt the system to a higher temperature and then quench it - this is done to ensure no particles are trapped 
4. Run the system at a constant number of particles (N), volume (V), and temperature (T) - NVT run 

After these steps are completed, the workflow will run analysis codes to derive properties of interest from the simulation output and trajectory files. Here, we are requesting to calculate:

1. Diffusion coefficients 
2. Radial distribution functions 

Similar to the DFT workflow examples, we will replace the actual MD jobs with fake ones. This will change the fireworks in the workflows so they will not actually run LAMMPS (which requires greater computer resources), but instead will simulate running LAMMPS by simply copying over pre-existing LAMMPS output and trajectory files. However, the overall steps in the workflow remain the same as if you are running the calculations on supercomputing resources. 

In [ ]:
wf = run_fake_lammps(
    wf,
    ref_dirs=[
        f"{ref_dir}/min",
        f"{ref_dir}/npt",
        f"{ref_dir}/melt",
        f"{ref_dir}/quench",
        f"{ref_dir}/nvt",
    ],
)

Now, we are ready to initialize the database and add the workflow to it. We will also get a summary of its status

In [ ]:
from datetime import date
from fireworks import LaunchPad

today = date.today()
d = today.strftime("%Y-%m-%d")
lpad = LaunchPad.auto_load()
lpad.reset(d)

lpad.add_wf(wf)
lpad.get_wf_summary_dict(fw_id=1)

Finally, we will run the workflow using rapidfile. This step will take a few moments. 

In [ ]:
from fireworks.core.rocket_launcher import rapidfire

rapidfire(lpad)

# System visualization

In [ ]:
from IPython.display import Image

Image(filename="/root/capsule/data/md/npt.gif", width=500, height=500)


# System properties

# Density

In [ ]:
%matplotlib inline

from pymatgen.io.lammps.outputs import parse_lammps_log
from mdproptools.utilities.fluctuations import plot_fluctuations

log = parse_lammps_log("/root/capsule/data/md/result/npt/log.lammps")

plot_fluctuations(log[0], "Density", "Density", "density.png", working_dir="/root/capsule/data/md/result")

**Note**: x-axis unit in the above plot should be seconds and the density on the y-axis is in g/cm3.

# Diffusivity

Diffusion is the movement of molecules from high concentration to low concentration regions. When two concentrations are different in two areas, this is referred to as the concentration gradient, moving concentration from high concentration to low concentration means the substance moves with its concentration gradient.

Diffusivity is a measure of the diffusion or transport of molecules or ions within the system. The higher the diffusivity of a specific molecule, the faster it moves throughout the solution. 

MISPR computes the diffusion of each specie in the system from the atomic coordinates at each time step. It outputs the diffusion coefficients in cm2/s. Let's take a look at the results. 

In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt 

df_diff = pd.read_csv("/root/capsule/data/md/result/analysis/diff/diffusion.csv",  index_col=0)
df_diff.columns = ["type", "diffusion (m2/s)", "std", "R2"]
df_diff 

# Radial distribution function

Radial distribution function (RDF) represents the probability of finding a particle within a distance of another particle. The higher this probability at short distance, the stronger the interaction is between the two particles. It is calculated using the atomic coordinates at each time step. 
RDF is one of the standard structural properties that can be predicted from MD simulations and it is critical for understanding the atomistic interactions in the system for correlating it to the overall observed behavior of the system. 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
df_rdf = pd.read_csv("/root/capsule/data/md/result/analysis/rdf/molecular/0.5/rdf.csv")

plt.plot(df_rdf["r ($\AA$)"], df_rdf["g_11-2"], label="C (CO2) - Propionitrile", color="black", linewidth=1.5)
#plt.plot(df_rdf["r ($\AA$)"], df_rdf["g_11-1"], label="C (CO2) - TBA", color="red", linewidth=1.5)
#plt.plot(df_rdf["r ($\AA$)"], df_rdf["g_11-3"], label="C (CO2) - BF4", color="green", linewidth=1.5)
plt.legend(fontsize=18, frameon=False)
plt.xlabel("distance (Angstroms)", fontsize=18)
plt.ylabel("RDF", fontsize=18)
plt.tick_params(axis='both', which='major', labelsize=16)
plt.show()